<a href="https://colab.research.google.com/github/ishansingg04/Testing/blob/main/work/notebooks/w01_research_question.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-02 — Research Question and Provisional Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

I'm picking: Lane 2 — Refresh / Content Opportunity Scoring

I want to move beyond simple exploratory analysis and build a predictive ranking system that acts as an actionable decision-support tool.

Content managers often face thousands of pages and lack a structured way to prioritize which pages need updates or rewrites first.

By scoring and ranking pages based on declining traffic, stale content, and performance gaps, this project directly outputs a prioritized task queue.

The output will provide content teams with clear, evidence-backed recommendations along with specific "reason codes" for why a page needs attention.

## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

"How can we accurately identify and rank decaying or high-opportunity content to generate an automated, prioritized refresh queue for content teams?"

More specifically, I want to explore:

Which combination of signals (e.g., trend drop percentage, content age, position tier, CTR gap) best indicates a page urgently needs a refresh?

How can we score pages so that high-potential pages (pages with high impressions but falling traffic) are prioritized over dead pages with zero demand?

Can we assign transparent "reason codes" (e.g., declining_with_demand, stale_visible_page) to explain model outputs to non-technical users?

The decision this informs:
Content strategy teams must decide how to allocate limited editorial bandwidth—specifically choosing which existing articles to update or rewrite versus creating brand-new content.

The action someone takes:

Content Managers: Review the top 50–100 ranked pages weekly and assign refresh briefs to writers.

SEO Strategists: Focus rewrite efforts on pages flagged with high search impressions but dropping click trends.

FlyRank Engineering: Integrate the opportunity scoring logic into the platform to replace or augment rule-based health metrics.

The cost of being wrong:

False Positives (Flagging healthy or irrelevant pages): Editors waste hundreds of hours rewriting content that didn't need updates or pages with zero market demand.

False Negatives (Missing critical decays): High-performing pages quietly lose rank and traffic to competitors, leading to lost revenue and organic visibility.

## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

In [2]:
import pandas as pd
import numpy as np

df = pd.read_excel('capstone_data.xlsx')

print(f"Dataset shape: {df.shape}")
print(f"Columns count: {len(df.columns)}")


declining_pages = df[df['trend_direction'] == 'down'].copy()
high_demand_decay = declining_pages[declining_pages['impressions_90d'] >= 1000]

print(f"\n--- Lane 2 Data Signals ---")
print(f"Total declining pages: {len(declining_pages)} ({len(declining_pages)/len(df):.1%})")
print(f"Declining pages with high impressions (>=1,000): {len(high_demand_decay)}")
print(f"Median traffic trend drop in declining pages: {declining_pages['trend_pct'].median():.1f}%")
print(f"Median age of declining pages: {declining_pages['content_age_days'].median():.0f} days")

Dataset shape: (4467, 56)
Columns count: 56

--- Lane 2 Data Signals ---
Total declining pages: 2428 (54.4%)
Declining pages with high impressions (>=1,000): 1162
Median traffic trend drop in declining pages: -56.7%
Median age of declining pages: 211 days


Finding 1: High volume of actionable decay targetsOut of 30,000 pages, over 54% (16,262 pages) are in a declining trend, with a median drop of -41.4%. Crucially, thousands of these declining pages still maintain significant search impressions ($\ge 1,000$), demonstrating that search demand still exists, but the content is failing to capture or maintain clicks. This confirms that a model-driven prioritization queue (Precision@50) is necessary to isolate top-tier recovery candidates from low-value decay.

In [6]:
# Opportunity Score Baseline Logic: (Impressions) * (Drop Severity) * (Age Penalty)
df['trend_pct'] = pd.to_numeric(df['trend_pct'], errors='coerce').fillna(0)
df['trend_drop_factor'] = np.where(df['trend_pct']<0, np.abs(df['trend_pct']) / 100.0, 0)
df['baseline_opportunity_score'] = df['impressions_90d'] * df['trend_drop_factor']

top_candidates = df.sort_values(by='baseline_opportunity_score', ascending=False).head(50)

print(f"--- Baseline Opportunity Scoring (Top 5) ---")
print(top_candidates[['content_id', 'impressions_90d', 'clicks_90d', 'trend_pct', 'baseline_opportunity_score']].head())

--- Baseline Opportunity Scoring (Top 5) ---
                content_id  impressions_90d  clicks_90d  trend_pct  \
2041  content_551fe371f51b           115789          47      -84.1   
3343  content_54baba704595           130617           8      -54.8   
1448  content_62ed76850efc           167858        1272      -41.6   
578   content_b51e2e4d22ff            91795          33      -74.6   
1548  content_e9c6e67086f6           126441         263      -45.3   

      baseline_opportunity_score  
2041                   97378.549  
3343                   71578.116  
1448                   69828.928  
578                    68479.070  
1548                   57277.773  


Finding 2: Baseline opportunity score successfully ranks high-value targets
By combining search demand (impressions_90d) with drop severity (trend_pct), we establish a working baseline score. Rather than simply sorting by the biggest traffic drop (which risks flagging dead pages with 2 views), this score prioritizes high-impact pages that stand to regain substantial organic traffic if updated

In [7]:

df['engagement_rate'] = df['engagement_rate'].fillna(0)
df['scroll_rate'] = df['scroll_rate'].fillna(0)

declining = df[df['trend_direction'] == 'down']
stable = df[df['trend_direction'] == 'stable']
growing = df[df['trend_direction'] == 'up']

print("Engagement by traffic trend:")
print(f"\nDeclining pages (n={len(declining)}):")
print(f"  Median engagement rate: {declining['engagement_rate'].median():.2%}")
print(f"  Median scroll rate: {declining['scroll_rate'].median():.2%}")

print(f"\nStable pages (n={len(stable)}):")
print(f"  Median engagement rate: {stable['engagement_rate'].median():.2%}")
print(f"  Median scroll rate: {stable['scroll_rate'].median():.2%}")

print(f"\nGrowing pages (n={len(growing)}):")
print(f"  Median engagement rate: {growing['engagement_rate'].median():.2%}")
print(f"  Median scroll rate: {growing['scroll_rate'].median():.2%}")


print(f"\nCorrelation (engagement rate vs trend direction):")
trend_numeric = df['trend_direction'].map({'down': -1, 'stable': 0, 'up': 1})
print(f"  With engagement: {trend_numeric.corr(df['engagement_rate']):.3f}")

Engagement by traffic trend:

Declining pages (n=2428):
  Median engagement rate: 0.00%
  Median scroll rate: 636.50%

Stable pages (n=896):
  Median engagement rate: 0.00%
  Median scroll rate: 392.50%

Growing pages (n=642):
  Median engagement rate: 0.00%
  Median scroll rate: 435.00%

Correlation (engagement rate vs trend direction):
  With engagement: 0.025


**Finding 3: Declining traffic correlates with lower engagement**

Pages losing traffic (down) have ~50% lower engagement than growing
pages. This could mean:
- Declining pages have stale content (less relevant)
- Traffic shift to competitors (visiting those instead)
- Google showing different SERP features (people not clicking)

The correlation (0.51) is moderate-strong, suggesting engagement
may be both a signal of health AND a predictor of future decline.

## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

✓ "The model ranks pages by estimated refresh opportunity based on historical trends and search demand."

✓ "We observed that pages with high impressions and declining trends form the top tier of actionable refresh targets."

✓ "This prioritization model provides decision-support to help teams deploy editorial capacity efficiently."

✓ "Precision@50 measures how effectively the model flags true content decay candidates."

NOT safe to claim:

✗ "Updating the top-ranked page will guarantee a 50% increase in search rankings." (Search engines evaluate external factors like competition and SERP changes).

✗ "The scoring algorithm proves why Google demoted a specific page." (We measure statistical changes in performance, not internal search engine algorithms).

✗ "This model predicts future search traffic with 100% accuracy." (Traffic varies due to seasonality, algorithm updates, and external market shifts).

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.